In [ ]:
import numpy as np
import pandas as pd
from google.colab import output
import copy
import time
import random

In [ ]:
# Configuración 1: Balanceada / Estándar
PESOS_CONFIG_1 = {
    "ganar_4": 1000,      # Prioridad máxima a ganar
    "bloquear_4": -900,   # Prioridad alta a evitar la derrota
    "ataque_3": 50,       # Desarrollo de ataques propios
    "prevenir_3": -40,    # Contención del desarrollo rival
    "centro": 10          # Control de las columnas centrales (3, 4, 5)
}

# Configuración 2: Agresiva / Control de Centro
PESOS_CONFIG_2 = {
    "ganar_4": 3000,      # Enfocada fuertemente en el remate
    "bloquear_4": -300,   # Menor atención a la defensa
    "ataque_3": 200,      # Incentivo alto a crear combinaciones de 3
    "prevenir_3": -10,    # Muy poca contención defensiva
    "centro": 50          # Alta valoración por dominar las columnas centrales
}

In [ ]:
class Ficha:
    simbolos = {
        "Rojo": "🔴",
        "Amarillo": "🟡"
    }

    def __init__(self, color):
        self.color = color
        self.simbolo = self.simbolos[color]

    def __str__(self):
        return self.color

    def __repr__(self):
        return str(self)

In [ ]:
class Tablero:
    def __init__(self, n_filas=6, n_columnas=9):
        self.n_filas = n_filas
        self.n_columnas = n_columnas
        self.grid = np.full((n_filas, n_columnas), fill_value=None, dtype=object)

    def colocar_ficha(self, ficha, columna):
        for fila in range(self.n_filas - 1, -1, -1):
            if self.grid[fila, columna] is None:
                self.grid[fila, columna] = ficha
                return True
        return False

    @staticmethod
    def verificar_fichas_son_iguales(array_fichas):
        return (
            len(set([None if ficha is None else ficha.color for ficha in array_fichas])) == 1
            and array_fichas[0] is not None
        )

    def verificar_victoria(self):
        # 1. Filas
        for fila in range(self.n_filas):
            for columna in range(self.n_columnas - 4):
                if self.verificar_fichas_son_iguales(self.grid[fila, columna:columna+5]):
                    return self.grid[fila, columna].color

        # 2. Columnas
        for fila in range(self.n_filas - 4):
            for columna in range(self.n_columnas):
                if self.verificar_fichas_son_iguales(self.grid[fila:fila+5, columna]):
                    return self.grid[fila, columna].color

        # 3. Diagonal descendente ( \ )
        for fila in range(self.n_filas - 4):
            for columna in range(self.n_columnas - 4):
                low = [self.grid[fila + i, columna + i] for i in range(5)]
                if self.verificar_fichas_son_iguales(low):
                    return self.grid[fila, columna].color

        # 4. Diagonal ascendente ( / )
        for fila in range(4, self.n_filas):
            for columna in range(self.n_columnas - 4):
                high = [self.grid[fila - i, columna + i] for i in range(5)]
                if self.verificar_fichas_son_iguales(high):
                    return self.grid[fila, columna].color

        return None

    def __str__(self):
        grid_str = ""
        for fila in range(self.n_filas):
            for columna in range(self.n_columnas):
                ficha = self.grid[fila, columna]
                if ficha is None:
                    grid_str += "▪️"
                else:
                    grid_str += ficha.simbolo + " "
            grid_str += "\n"
        return grid_str

    def __repr__(self):
        return str(self)

    def esta_lleno(self):
        return all(self.grid[0, col] is not None for col in range(self.n_columnas))

    def columnas_validas(self):
        return [col for col in range(self.n_columnas) if self.grid[0, col] is None]

    def clonar(self):
        return copy.deepcopy(self)

    def _obtener_ventanas(self):
        ventanas = []
        # Horizontales
        for f in range(self.n_filas):
            for c in range(self.n_columnas - 4):
                ventanas.append([self.grid[f, c + i] for i in range(5)])
        # Verticales
        for f in range(self.n_filas - 4):
            for c in range(self.n_columnas):
                ventanas.append([self.grid[f + i, c] for i in range(5)])
        # Diagonales (\)
        for f in range(self.n_filas - 4):
            for c in range(self.n_columnas - 4):
                ventanas.append([self.grid[f + i, c + i] for i in range(5)])
        # Diagonales (/)
        for f in range(4, self.n_filas):
            for c in range(self.n_columnas - 4):
                ventanas.append([self.grid[f - i, c + i] for i in range(5)])
        return ventanas

    def evaluar_heuristica(self, color_jugador, tipo_heuristica="todas", pesos=None):
        P = pesos if pesos is not None else PESOS_CONFIG_1
        color_oponente = "Amarillo" if color_jugador == "Rojo" else "Rojo"
        puntaje_total = 0

        # H1: Control del Centro (Columnas 3, 4, 5)
        if tipo_heuristica in ["h1", "todas"]:
            for col in [3, 4, 5]:
                fichas_centro = [self.grid[f, col] for f in range(self.n_filas)]
                propias = sum(1 for f in fichas_centro if f is not None and f.color == color_jugador)
                oponentes = sum(1 for f in fichas_centro if f is not None and f.color == color_oponente)
                puntaje_total += (propias * P["centro"]) - (oponentes * P["centro"])

        # Evaluador de ventanas
        def evaluar_ventana(ventana):
            puntaje = 0
            propias = sum(1 for f in ventana if f is not None and f.color == color_jugador)
            oponentes = sum(1 for f in ventana if f is not None and f.color == color_oponente)
            vacios = sum(1 for f in ventana if f is None)

            # H2: Ataque Inminente (4 propias + 1 libre)
            if tipo_heuristica in ["h2", "todas"] and propias == 4 and vacios == 1:
                puntaje += P["ganar_4"]

            # H3: Bloqueo Crítico (4 oponente + 1 libre)
            if tipo_heuristica in ["h3", "todas"] and oponentes == 4 and vacios == 1:
                puntaje += P["bloquear_4"]

            # H4: Construcción de Ataque (3 propias + 2 libres)
            if tipo_heuristica in ["h4", "todas"] and propias == 3 and vacios == 2:
                puntaje += P["ataque_3"]

            # H5: Prevención de Ataque Rival (3 oponente + 2 libres)
            if tipo_heuristica in ["h5", "todas"] and oponentes == 3 and vacios == 2:
                puntaje += P["prevenir_3"]

            return puntaje

        for v in self._obtener_ventanas():
            puntaje_total += evaluar_ventana(v)

        return puntaje_total

In [ ]:
class TimeoutException(Exception):
    """Excepción para cortar la búsqueda iterativa cuando el tiempo se agota."""
    pass


class Nodo:
    def __init__(self, estado, padre=None, accion=None, profundidad=0, tipo_nodo="MAX"):
        self.estado = estado
        self.padre = padre
        self.accion = accion
        self.profundidad = profundidad
        self.tipo_nodo = tipo_nodo  # "MAX" o "MIN"
        self.hijos = []
        self.valor_evaluado = None

    def expandir(self, color_max, color_min):
        self.hijos = []
        color_a_colocar = color_max if self.tipo_nodo == "MAX" else color_min
        siguiente_tipo = "MIN" if self.tipo_nodo == "MAX" else "MAX"

        for col in self.estado.columnas_validas():
            tablero_hijo = self.estado.clonar()
            tablero_hijo.colocar_ficha(Ficha(color_a_colocar), col)

            nodo_hijo = Nodo(
                estado=tablero_hijo,
                padre=self,
                accion=col,
                profundidad=self.profundidad + 1,
                tipo_nodo=siguiente_tipo
            )
            self.hijos.append(nodo_hijo)

        return self.hijos

    def mostrar_arbol(self, max_prof=1, indent=""):
        if self.profundidad > max_prof:
            return

        info_accion = f"Jugada en Columna {self.accion}" if self.accion is not None else "Estado Inicial (Raíz)"
        info_valor = f" | Utilidad Minimax: {self.valor_evaluado}" if self.valor_evaluado is not None else ""

        print(f"{indent} Profundidad {self.profundidad} [{self.tipo_nodo}] - {info_accion}{info_valor}")

        lineas_tablero = str(self.estado).split('\n')
        for linea in lineas_tablero:
            if linea.strip():
                print(f"{indent}  {linea}")

        for hijo in self.hijos:
            hijo.mostrar_arbol(max_prof, indent + "    ")

In [ ]:
class Minimax_Solver:
    def __init__(self, player_color, max_time=2.0, heuristica="todas", pesos=None):
        self.player_color = player_color
        self.max_time = max_time
        self.heuristica = heuristica
        self.pesos = pesos
        self.time_start = None
        self.oponente_color = "Amarillo" if player_color == "Rojo" else "Rojo"
        self.ultimo_arbol_raiz = None

        # Métricas para benchmark
        self.nodos_expandidos = 0
        self.max_profundidad_alcanzada = 0
        self.tiempo_ultima_jugada = 0

    def encontrar_mejor_columna(self, tablero):
        self.time_start = time.time()
        self.nodos_expandidos = 0
        self.max_profundidad_alcanzada = 0

        columnas_validas = tablero.columnas_validas()
        if not columnas_validas or len(columnas_validas) == 1:
            self.tiempo_ultima_jugada = time.time() - self.time_start
            return columnas_validas[0] if columnas_validas else None

        mejor_accion_global = columnas_validas[0]
        profundidad_max = 1

        while True:
            if time.time() - self.time_start >= self.max_time:
                break

            try:
                nodo_raiz = Nodo(estado=tablero.clonar(), profundidad=0, tipo_nodo="MAX")
                utility, accion_nivel = self.__maximize(
                    nodo_raiz, -float('inf'), float('inf'), limite_profundidad=profundidad_max
                )

                if accion_nivel is not None:
                    mejor_accion_global = accion_nivel
                    nodo_raiz.valor_evaluado = utility
                    self.ultimo_arbol_raiz = nodo_raiz
                    self.max_profundidad_alcanzada = profundidad_max

                profundidad_max += 1

            except TimeoutException:
                break

        self.tiempo_ultima_jugada = time.time() - self.time_start
        return mejor_accion_global

    def __verificar_tiempo(self):
        if time.time() - self.time_start >= self.max_time:
            raise TimeoutException()

    def __maximize(self, nodo, alpha, beta, limite_profundidad):
        self.__verificar_tiempo()

        ganador = nodo.estado.verificar_victoria()
        if ganador == self.player_color:
            nodo.valor_evaluado = 100000
            return 100000, nodo.accion
        elif ganador == self.oponente_color:
            nodo.valor_evaluado = -100000
            return -100000, nodo.accion

        if nodo.estado.esta_lleno() or nodo.profundidad >= limite_profundidad:
            val = nodo.estado.evaluar_heuristica(self.player_color, tipo_heuristica=self.heuristica, pesos=self.pesos)
            nodo.valor_evaluado = val
            return val, nodo.accion

        max_utility = -float('inf')
        mejor_accion = None
        nodos_hijos = nodo.expandir(self.player_color, self.oponente_color)
        self.nodos_expandidos += len(nodos_hijos)

        for hijo in nodos_hijos:
            utility, _ = self.__minimize(hijo, alpha, beta, limite_profundidad)
            if utility > max_utility:
                max_utility = utility
                mejor_accion = hijo.accion

            if max_utility >= beta:
                break  # Poda Beta

            alpha = max(alpha, max_utility)

        nodo.valor_evaluado = max_utility
        return max_utility, mejor_accion

    def __minimize(self, nodo, alpha, beta, limite_profundidad):
        self.__verificar_tiempo()

        ganador = nodo.estado.verificar_victoria()
        if ganador == self.oponente_color:
            nodo.valor_evaluado = -100000
            return -100000, nodo.accion
        elif ganador == self.player_color:
            nodo.valor_evaluado = 100000
            return 100000, nodo.accion

        if nodo.estado.esta_lleno() or nodo.profundidad >= limite_profundidad:
            val = nodo.estado.evaluar_heuristica(self.player_color, tipo_heuristica=self.heuristica, pesos=self.pesos)
            nodo.valor_evaluado = val
            return val, nodo.accion

        min_utility = float('inf')
        mejor_accion = None
        nodos_hijos = nodo.expandir(self.player_color, self.oponente_color)
        self.nodos_expandidos += len(nodos_hijos)

        for hijo in nodos_hijos:
            utility, _ = self.__maximize(hijo, alpha, beta, limite_profundidad)
            if utility < min_utility:
                min_utility = utility
                mejor_accion = hijo.accion

            if min_utility <= alpha:
                break  # Poda Alpha

            beta = min(beta, min_utility)

        nodo.valor_evaluado = min_utility
        return min_utility, mejor_accion

In [ ]:
class Jugador:
    def __init__(self, nombre, color):
        self.nombre = nombre
        self.color = color

    def seleccionar_columna(self, tablero):
        pass

In [ ]:
class Humano(Jugador):
    def seleccionar_columna(self, tablero):
        columnas_validas = tablero.columnas_validas()
        while True:
            columna = input(f"[{self.nombre}] Selecciona la columna: ")
            if columna.isnumeric():
                col_int = int(columna)
                if 0 <= col_int < tablero.n_columnas:
                    if col_int in columnas_validas:
                        return col_int
                    else:
                        print("Columna llena.")
                else:
                    print(f"Fuera de rango (0 a {tablero.n_columnas - 1}).")
            else:
                print("Entrada inválida.")

In [ ]:
class RandomBot(Jugador):
    def seleccionar_columna(self, tablero):
        validas = tablero.columnas_validas()
        return random.choice(validas) if validas else 0

In [ ]:
class GreedyBot(Jugador):
    def seleccionar_columna(self, tablero):
        mejor_puntaje = -float('inf')
        validas = tablero.columnas_validas()
        mejor_col = random.choice(validas)

        for col in validas:
            temp = tablero.clonar()
            temp.colocar_ficha(Ficha(self.color), col)

            if temp.verificar_victoria() == self.color:
                return col  # Ganar de inmediato

            puntaje = temp.evaluar_heuristica(self.color)
            if puntaje > mejor_puntaje:
                mejor_puntaje = puntaje
                mejor_col = col

        return mejor_col

In [ ]:
class PeorBot(Jugador):
    def seleccionar_columna(self, tablero):
        peor_puntaje = float('inf')
        validas = tablero.columnas_validas()
        peor_col = random.choice(validas)

        for col in validas:
            temp = tablero.clonar()
            temp.colocar_ficha(Ficha(self.color), col)

            puntaje = temp.evaluar_heuristica(self.color)
            if puntaje < peor_puntaje:
                peor_puntaje = puntaje
                peor_col = col

        return peor_col

In [ ]:
class MinimaxBot(Jugador):
    def __init__(self, nombre, color, max_time=2.0, heuristica="todas", pesos=None):
        super().__init__(nombre, color)
        self.solver = Minimax_Solver(color, max_time=max_time, heuristica=heuristica, pesos=pesos)

    def seleccionar_columna(self, tablero):
        return self.solver.encontrar_mejor_columna(tablero)

    def imprimir_ultimo_arbol(self, max_profundidad=2):
        if self.solver.ultimo_arbol_raiz is not None:
            print(f"\nÁrbol de búsqueda generado por {self.nombre} (Última jugada)")
            self.solver.ultimo_arbol_raiz.mostrar_arbol(max_prof=max_profundidad)
        else:
            print("Aún no se ha generado ningún árbol de búsqueda.")

In [ ]:
class MenuInteractivo:
    def __init__(self):
        self.jugadores_partida = []

    def configurar_jugador(self, index, color):
        nombre = input(f"Nombre del Jugador {index} ({color}): ") or f"J{index}"
        print(f"Tipo de jugador para {nombre}:")
        print("1. Humano")
        print("2. Minimax (IA)")
        print("3. Greedy (Avaro)")
        print("4. Random (Aleatorio)")
        print("5. Peor Decisión")

        tipo = input("Selecciona (1-5): ").strip()

        if tipo == "2":
            tiempo = float(input("Tiempo máximo para la IA en segundos (ej: 2.0): ") or 2.0)

            # Selección de Heurística
            print("\nSelecciona la Heurística para la IA:")
            print("1. Control del centro (h1)")
            print("2. Ataque inminente (h2)")
            print("3. Bloqueo crítico (h3)")
            print("4. Construcción de ataque (h4)")
            print("5. Prevención de ataque rival (h5)")
            print("6. Todas las heurísticas combinadas")

            opc_h = input("Selecciona (1-6): ").strip()
            mapa_h = {"1": "h1", "2": "h2", "3": "h3", "4": "h4", "5": "h5", "6": "todas"}
            heuristica_elegida = mapa_h.get(opc_h, "todas")

            # Selección de Configuración de Pesos
            print("\nConfiguración de Pesos:")
            print("1. Balanceada (Estándar)")
            print("2. Agresiva (Ataque y Centro)")
            cfg = input("Selecciona (1-2): ").strip()

            pesos = PESOS_CONFIG_1 if cfg != "2" else PESOS_CONFIG_2
            return MinimaxBot(nombre, color, max_time=tiempo, heuristica=heuristica_elegida, pesos=pesos)

        elif tipo == "3":
            return GreedyBot(nombre, color)
        elif tipo == "4":
            return RandomBot(nombre, color)
        elif tipo == "5":
            return PeorBot(nombre, color)
        else:
            return Humano(nombre, color)

    def jugar_partida(self, mostrar_UI=True):
        tablero = Tablero()
        turno = 0

        while True:
            jugador_actual = self.jugadores_partida[turno % 2]
            if mostrar_UI:
                output.clear()
                print(f"Turno de: {jugador_actual.nombre} ({jugador_actual.color})\n")
                print(tablero)

            col = jugador_actual.seleccionar_columna(tablero)
            tablero.colocar_ficha(Ficha(jugador_actual.color), col)

            ganador = tablero.verificar_victoria()
            if ganador is not None or tablero.esta_lleno():
                if mostrar_UI:
                    output.clear()
                    print(tablero)
                    if ganador:
                        print(f"\n Ganó {jugador_actual.nombre}")
                    else:
                        print("\n Empate ")
                return ganador, tablero

            turno += 1

    def iniciar_benchmark(self):
        print("Iniciando Benchmark Automatizado...")
        resultados = []
        oponentes = [
            ("Random", RandomBot("RND", "Amarillo")),
            ("Greedy", GreedyBot("GRD", "Amarillo")),
            ("Peor", PeorBot("WORST", "Amarillo"))
        ]

        for nombre_oponente, bot_oponente in oponentes:
            ia_principal = MinimaxBot("Minimax-Main", "Rojo", max_time=1.0, pesos=PESOS_CONFIG_1)
            self.jugadores_partida = [ia_principal, bot_oponente]

            ganador, _ = self.jugar_partida(mostrar_UI=False)

            nodos = ia_principal.solver.nodos_expandidos
            prof = ia_principal.solver.max_profundidad_alcanzada
            tiempo_prom = ia_principal.solver.tiempo_ultima_jugada
            puntos = ia_principal.solver.ultimo_arbol_raiz.valor_evaluado if ia_principal.solver.ultimo_arbol_raiz else 0

            quien_gano = "Minimax" if ganador == "Rojo" else ("Empate" if not ganador else nombre_oponente)

            resultados.append({
                "Contrincante": nombre_oponente,
                "Ganador": quien_gano,
                "Puntos / Utilidad": puntos,
                "Nodos Expandidos (Últ. Jugada)": nodos,
                "Profundidad (Últ. Jugada)": prof,
                "Tiempo (s)": round(tiempo_prom, 3)
            })

        df = pd.DataFrame(resultados)
        display(df)

    def menu_principal(self):
        print("1. Jugar partida manual")
        print("2. Ejecutar Benchmark (Tabla comparativa automática)")
        opcion = input("Elige una opción: ").strip()

        if opcion == "2":
            self.iniciar_benchmark()
        else:
            self.jugadores_partida = [
                self.configurar_jugador(1, "Rojo"),
                self.configurar_jugador(2, "Amarillo")
            ]
            self.jugar_partida(mostrar_UI=True)


# Ejecución
juego = MenuInteractivo()
juego.menu_principal()

1. Jugar partida manual
2. Ejecutar Benchmark (Tabla comparativa automática)
Elige una opción: 2
Iniciando Benchmark Automatizado...


,Contrincante,Ganador,Puntos / Utilidad,Nodos Expandidos (Últ. Jugada),Profundidad (Últ. Jugada),Tiempo (s)
0,Random,Minimax,100000,2268,5,1.001
1,Greedy,Minimax,100000,1616,11,1.000
2,Peor,Minimax,100000,1785,5,1.114


In [ ]:
# Inspeccionar el árbol generado en la última jugada por cualquier MinimaxBot
print("   INSPECCIÓN DE TABLEROS EN EL ÁRBOL DE BÚSQUEDA")

arboles_encontrados = False

if 'juego' in locals() and hasattr(juego, 'jugadores_partida'):
    for jugador in juego.jugadores_partida:
        if isinstance(jugador, MinimaxBot):
            arboles_encontrados = True
            jugador.imprimir_ultimo_arbol(max_profundidad=7)

if not arboles_encontrados:
    print("No se encontraron bots de IA con árboles guardados. Ejecuta una partida primero.")

   INSPECCIÓN DE TABLEROS EN EL ÁRBOL DE BÚSQUEDA

Árbol de búsqueda generado por Minimax-Main (Última jugada)
 Profundidad 0 [MAX] - Estado Inicial (Raíz) | Utilidad Minimax: 100000
  🟡 🔴 ▪️▪️▪️▪️▪️▪️▪️
  🔴 🟡 ▪️▪️▪️▪️▪️▪️▪️
  🟡 🔴 ▪️▪️▪️▪️▪️▪️▪️
  🔴 🔴 ▪️▪️▪️▪️🟡 ▪️▪️
  🟡 🟡 ▪️🔴 ▪️🟡 🟡 ▪️▪️
  🟡 🟡 ▪️🔴 🔴 🔴 🔴 ▪️▪️
     Profundidad 1 [MIN] - Jugada en Columna 2 | Utilidad Minimax: 100000
      🟡 🔴 ▪️▪️▪️▪️▪️▪️▪️
      🔴 🟡 ▪️▪️▪️▪️▪️▪️▪️
      🟡 🔴 ▪️▪️▪️▪️▪️▪️▪️
      🔴 🔴 ▪️▪️▪️▪️🟡 ▪️▪️
      🟡 🟡 ▪️🔴 ▪️🟡 🟡 ▪️▪️
      🟡 🟡 🔴 🔴 🔴 🔴 🔴 ▪️▪️
     Profundidad 1 [MIN] - Jugada en Columna 3 | Utilidad Minimax: 100000
      🟡 🔴 ▪️▪️▪️▪️▪️▪️▪️
      🔴 🟡 ▪️▪️▪️▪️▪️▪️▪️
      🟡 🔴 ▪️▪️▪️▪️▪️▪️▪️
      🔴 🔴 ▪️🔴 ▪️▪️🟡 ▪️▪️
      🟡 🟡 ▪️🔴 ▪️🟡 🟡 ▪️▪️
      🟡 🟡 ▪️🔴 🔴 🔴 🔴 ▪️▪️
         Profundidad 2 [MAX] - Jugada en Columna 2 | Utilidad Minimax: 100000
          🟡 🔴 ▪️▪️▪️▪️▪️▪️▪️
          🔴 🟡 ▪️▪️▪️▪️▪️▪️▪️
          🟡 🔴 ▪️▪️▪️▪️▪️▪️▪️
          🔴 🔴 ▪️🔴 ▪️▪️🟡 ▪️▪️
          🟡 🟡 ▪️🔴 ▪️🟡 🟡 ▪️▪️
          🟡 🟡 🟡 🔴 🔴 